# CENG463 PA2

In this programming assignment, you will be dealing with word embeddings and neural networks. You will use Python for this task. You can use libraries such as `pandas`, `nltk`, `numpy` etc. for your implementations, or implement your own functions. However, you are expected to analyse and reason about your implementation and results. The assignment consists of 3 questions.

### IMPORTANT NOTE

Do not move or delete the given cells, only add cells inbetween the questions for your answers.

In [ ]:
# UPDATE THIS CELL TO INSTALL NEEDED LIBRARIES.
# MAKE SURE TO ADD EVERYTHING THAT NEEDS TO BE INSTALLED IN THIS CELL!

# we will use pip to install packages - you can add others below
!pip install pandas
!pip install numpy
!pip install nltk
!pip install gensim
!pip install scikit-learn
!pip install tensorflow
!pip install regex

# and import them here - you can add others below
import pandas as pd
import numpy as np
import nltk
import gensim
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Concatenate, Dropout
)

#for metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


# for pa1
import re
import time
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer

## Q1 - Word embeddings (50 points)

In this question, you will first train a Word2Vec model, then use it to represent and reason about user reviews.

### Q1.A - training (10 points)
Load the `user_review_train.csv` file shared with you. Using `Word2Vec` module of `gensim.models`, train a **skip-gram** Word2Vec model on the train data.

#### Notes and tips

- Use the given preprocessing function `preprocess_review`.

In [2]:
# PREPROCESSING FUNCTIONS GIVEN FOR YOU

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')  
nltk.download('stopwords')
nltk.download('punkt_tab')

def preprocess_review(review):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    sentences = sent_tokenize(review)
    
    lemmatized_review = []
    
    for sentence in sentences:
        tokenized_sentence = word_tokenize(sentence)
        lowercased_sentence = [token.lower() for token in tokenized_sentence]
        stopwords_removed_sentence = [token for token in lowercased_sentence if token not in stop_words]
        lemmatized_sentence = [lemmatizer.lemmatize(token) for token in stopwords_removed_sentence]
        
        lemmatized_review = lemmatized_review + lemmatized_sentence
    
    return lemmatized_review

[nltk_data] Downloading package wordnet to /Users/ovak/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /Users/ovak/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /Users/ovak/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
# Q1.A - implementation
# you can add cells below if needed
review_test = pd.read_csv('data/user_review_train.csv')
review_test["preprocessed_review"] = review_test["review"].apply(preprocess_review)
print(review_test["preprocessed_review"].head())

model = gensim.models.Word2Vec(sentences=review_test["preprocessed_review"], 
                               vector_size=100, 
                               window=5, 
                               min_count=5,
                               workers=4, 
                               sg=1,
                               epochs=10)
print("Word2Vec model trained successfully.")
print(f"Vocabulary size: {len(model.wv)}")

0    [worst, mobile, bought, ever, ,, battery, drai...
1    [worst, phone, everthey, changed, last, phone,...
2    ['m, telling, n't, buyi, 'm, totally, disappoi...
3                               [battery, level, worn]
4    ['s, hitting, problem, ..., phone, hanging, pr...
Name: preprocessed_review, dtype: object


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec model trained successfully.
Vocabulary size: 2539


### Q1.B - word similarity (10 points)

Using the trained model, report the following:

- Similarity between "good" and "bad"
- Similar words to "good"
- Similar words to "bad"
- Similar words to "good" but not similar to "bad"
- Similar words to "good" but not similar to "bad"

and discuss the reported words and scores. Is it possible to identify specific good/bad features of the product that is being reviewed? What other words can be looked up to get more insight?

#### Notes and tips

- Check the [documentation](https://tedboy.github.io/nlps/generated/generated/gensim.models.Word2Vec.html) of `gensim.models.Word2Vec` to find relevant methods.

In [17]:
# Q1.B - implementation
# you can add cells below if needed

sim = model.wv.similarity('good', 'bad')  # Example similarity check
print("Similarity between 'good' and 'bad':", sim)

most_similar_words = model.wv.most_similar("good")
print("Words most similar to 'good':", most_similar_words)

most_similar_words = model.wv.most_similar("bad")
print("Words most similar to 'bad':", most_similar_words)


simGoodbutNotBad = model.wv.most_similar(positive=['good'], negative=['bad'], topn=5)
print("Words similar to 'good' but not 'bad':", simGoodbutNotBad)

simGoodbutNotBad = model.wv.most_similar(positive=['bad'], negative=['good'], topn=5)
print("Words similar to 'bad' but not 'good':", simGoodbutNotBad)

Similarity between 'good' and 'bad': 0.499957
Words most similar to 'good': [('nice', 0.8183974027633667), ('satisfying', 0.8095851540565491), ('good.overall', 0.7919037342071533), ('awsome', 0.7775800228118896), ('beautiful', 0.7709921002388), ('gr8', 0.762909471988678), ('good.but', 0.760248064994812), ('satisfactory', 0.7562212944030762), ('awesome', 0.7509245872497559), ('unbeatable', 0.7498699426651001)]
Words most similar to 'bad': [('poor', 0.6652644276618958), ('pathetic', 0.6275802850723267), ('worst', 0.6212835907936096), ('👎', 0.6079564094543457), ('nd', 0.5951585173606873), ('amozon', 0.5911490321159363), ('bed', 0.5847691893577576), ('wrost', 0.5844359397888184), ('vary', 0.5841897130012512), ('lose', 0.5730414390563965)]
Words similar to 'good' but not 'bad': [('nice', 0.4199712872505188), ('operation', 0.4105502963066101), ('wise', 0.40016183257102966), ('concern', 0.38534700870513916), ('costly', 0.3848535716533661)]
Words similar to 'bad' but not 'good': [('reply', 0.2

### Seem we cannot get a good results to detect good and bad features. Let's try another pair like worst and best.

In [25]:
sim = model.wv.similarity('worst', 'best')  # Example similarity check
print("Similarity between 'worst' and 'best':", sim)

Similarity between 'worst' and 'best': 0.517487


In [26]:
most_similar_words = model.wv.most_similar("worst")
print("Words most similar to 'worst':", most_similar_words)

most_similar_words = model.wv.most_similar("best")
print("Words most similar to 'best':", most_similar_words)


simGoodbutNotBad = model.wv.most_similar(positive=['worst'], negative=['best'], topn=5)
print("Words similar to 'worst' but not 'best':", simGoodbutNotBad)

simGoodbutNotBad = model.wv.most_similar(positive=['best'], negative=['worst'], topn=5)
print("Words similar to 'bad' but not 'best':", simGoodbutNotBad)

Words most similar to 'worst': [('wrost', 0.6994557976722717), ('seen', 0.6964107751846313), ('worse', 0.6893751621246338), ('pathetic', 0.6570553183555603), ('ever', 0.6484885811805725), ('disgusting', 0.6456800699234009), ('worest', 0.6428794860839844), ('worthless', 0.6347588300704956), ('utter', 0.6287044882774353), ('world', 0.6234968900680542)]
Words most similar to 'best': [('category', 0.7451916933059692), ('15000', 0.7446299195289612), ('affordable', 0.7421995401382446), ('osm', 0.7356728315353394), ('awesome', 0.7226203083992004), ('15k', 0.7172533273696899), ('budget', 0.7081776857376099), ('prize', 0.7071000933647156), ('awsome', 0.7013788819313049), ('lovely', 0.699744701385498)]
Words similar to 'worst' but not 'best': [('responding', 0.42473074793815613), ('respond', 0.3683648109436035), ('several', 0.34905320405960083), ('stop', 0.3474757671356201), ('called', 0.3449437916278839)]
Words similar to 'bad' but not 'best': [('budget', 0.47222402691841125), ('range', 0.42063

### Q1.B - discussion
Write your discussion here

### Q1.C - representation (15 points)

An important use of word embeddings is representing "documents" (reviews in our case). For this question, before creating the representations, do the following:

- Randomly sample 2 reviews from sentiment label 0, refer to them as sent0_a and sent0_b.
- Randomly sample 2 reviews from sentiment label 1, refer to them as sent1_a and sent1_b.

After the sampling, follow these steps to represent each review:

- Preprocess the review with the given `preprocess_review` function.
- For each token in the review, fetch the vector of that token.
- Take the average of the token vectors in the review to represent that review.

Then, calculate and report the cosine similarity of the two vectors representing:
    - sent0_a and sent0_b
    - sent0_a and sent1_a
    - sent1_a and sent1_b

Does this representation work to capture the labels of the reviews? Do you think there is a better way to represent each review instead of taking the average of the word vectors? Discuss your findings with respect to these questions. Repeating the sampling process several times might give you a better insight.

#### Notes and tips

- You can use `numpy` for your calculations.

In [39]:
# Q1.C - implementation
# you can add cells below if needed
def get_rndm_review(sentiment):
    while True:
        index = np.random.randint(0, len(review_test))
        select_review = review_test["review"][index]
        if(review_test["sentiment"][index] == sentiment):
            print("Selected review index:", index)
            break
    return select_review


np.random.seed(33)
sent1_a = get_rndm_review(1)
sent1_b = get_rndm_review(1)
sent0_a = get_rndm_review(0)
sent0_b = get_rndm_review(0)

print("Positive Review 1:", sent1_a)
print("Positive Review 2:", sent1_b)      
print("Negative Review 1:", sent0_a)
print("Negative Review 2:", sent0_b)

Selected review index: 10898
Selected review index: 11465
Selected review index: 57
Selected review index: 102
Positive Review 1: Good I like you
Positive Review 2: Very good
Negative Review 1: It's battery is draining very fast.
Negative Review 2: Poor battery and a lot of heating even for normal usage. I'll prefer Moto G5 compare to this.


In [40]:
#preprocess reviews

preprocessed_positive_review_1 = preprocess_review(sent1_a)
preprocessed_positive_review_2 = preprocess_review(sent1_b)
preprocessed_negative_review_1 = preprocess_review(sent0_a)
preprocessed_negative_review_2 = preprocess_review(sent0_b)

print("Preprocessed Positive Review 1:", preprocessed_positive_review_1)
print("Preprocessed Positive Review 2:", preprocessed_positive_review_2)
print("Preprocessed Negative Review 1:", preprocessed_negative_review_1)
print("Preprocessed Negative Review 2:", preprocessed_negative_review_2)


Preprocessed Positive Review 1: ['good', 'like']
Preprocessed Positive Review 2: ['good']
Preprocessed Negative Review 1: ["'s", 'battery', 'draining', 'fast', '.']
Preprocessed Negative Review 2: ['poor', 'battery', 'lot', 'heating', 'even', 'normal', 'usage', '.', "'ll", 'prefer', 'moto', 'g5', 'compare', '.']


In [41]:
#For each token in the review, fetch the vector of that token.

def get_review_vector(review):
    review_vector = np.zeros((100,))
    valid_word_count = 0
    for token in review:
        if token in model.wv:  # Check if token exists
            review_vector += model.wv[token]
            valid_word_count += 1
    
    if valid_word_count > 0:  # Avoid division by zero
        review_vector /= valid_word_count
    return review_vector

positive_review_vector_1 = get_review_vector(preprocessed_positive_review_1)
positive_review_vector_2 = get_review_vector(preprocessed_positive_review_2)
negative_review_vector_1 = get_review_vector(preprocessed_negative_review_1)
negative_review_vector_2 = get_review_vector(preprocessed_negative_review_2)

print("Positive Review Vector 1:", positive_review_vector_1)
print("Positive Review Vector 2:", positive_review_vector_2)
print("Negative Review Vector 1:", negative_review_vector_1)
print("Negative Review Vector 2:", negative_review_vector_2)

Positive Review Vector 1: [ 1.10560663e-01  4.74479683e-01 -3.87284718e-02 -7.21090708e-02
  6.04379899e-02 -2.36155540e-02  3.62363636e-01  4.37541351e-01
  9.12892930e-02 -3.63050029e-01  2.42754996e-01 -4.34016924e-01
  1.20959006e-01  1.02862734e-02  1.61734264e-01 -2.49262378e-01
  2.16455437e-01  8.90580080e-02 -3.48619014e-01 -2.36611155e-01
  7.59679824e-04  5.37809376e-02 -1.44323781e-02 -2.47160830e-01
 -3.01458567e-01 -2.18233582e-01 -3.42036888e-01 -3.36662039e-01
 -2.81456145e-02 -6.60371892e-02  1.59279950e-01 -7.40569495e-02
  1.41784359e-01 -2.38756180e-01 -1.86865143e-02  1.76035858e-01
 -5.89391440e-02 -1.82166421e-01 -6.78933477e-02 -4.00094055e-01
 -9.41093266e-02 -6.12154463e-02  2.88764015e-04 -1.40060913e-01
  4.88385320e-01 -9.65299178e-03 -1.59435395e-01 -2.84817003e-01
  2.94684507e-02  1.02664229e-01  1.92714870e-01 -2.66144078e-01
 -3.88582423e-02 -1.06974760e-01 -7.23895803e-02 -2.08165906e-01
  2.37359963e-01 -1.81263633e-01 -5.91383926e-02 -8.62644985e-03

In [42]:
# Cosine similarty function

def cosine_similarity(vec1, vec2):
    cos_sim = np.dot(vec1, vec2)/(np.linalg.norm(vec1)*np.linalg.norm(vec2))
    return cos_sim

# Calculate cosine similarities
sim_pos_pos = cosine_similarity(positive_review_vector_1, positive_review_vector_2)
sim_pos_neg_1 = cosine_similarity(positive_review_vector_1, negative_review_vector_1)
sim_neg_neg = cosine_similarity(negative_review_vector_1, negative_review_vector_2)


print("Cosine Similarity between two positive reviews:", sim_pos_pos)
print("Cosine Similarity between positive and negative review:", sim_pos_neg_1)
print("Cosine Similarity between two negative reviews:", sim_neg_neg)


Cosine Similarity between two positive reviews: 0.7651455583004072
Cosine Similarity between positive and negative review: 0.5957894674763837
Cosine Similarity between two negative reviews: 0.76285493026921


In [ ]:
# alternative approach 

### Q1.C - discussion
Write your discussion here

### Q1.D - training and comparing classifiers (15 points)

For this task, you will use the `user_review_train.csv` and `user_review_test.csv` files to train a binary classification model with Word2Vec representations, and compare its performance with a binary classifier using Bag-of-Words representation.

As the Bag-of-Words classifier, you can either choose the best performing classifier you have implemented in Question 3 of Programming Assignment 1, or you can follow these steps:

- Preprocess the review with the given `preprocess_review` function.
- Order all unique tokens by frequency, take the most frequent 100.
- Use these 100 words as the corpus for Bag-of-Words representation.

For the Word2Vec model, represent the reviews by following these steps:

- Preprocess the review with the given `preprocess_review` function.
- For each token in the review that is also in the most frequent 100 tokens, fetch the vector of that token.
- Take the average of the token vectors selected to represent that review.

After training both classifiers on `user_review_train.csv`, test them with `user_review_test.csv` and report the performance of your models with four metrics: accuracy, precision, recall and F1-score. Compare the performance of both models and discuss in detail.

#### Notes and tips

- You can use `CountVectorizer` from `scikit-learn` or any other library available for Bag-of-Words representation.
- You should select a classification method from the following set of classifiers: `[Naive Bayes, Support Vector Machine, Logistic Regression, Random Forest]`. You can use `scikit-learn`, `nltk`, or any other library for the classifier implementations. 
- You should **not** use the test set `user_reviews_test.csv` during your training process. You should use `user_reviews_train.csv` only.
- You may add a validation step in your training process. To do this, you can further split the `user_reviews_train.csv` data and apply k-fold cross validation.

### TAKEN FROM MY PA1 PIECE OF CODE

In [9]:
# Preprocessing helper functions


train_data = pd.read_csv('data/user_review_train.csv')
test_data = pd.read_csv('data/user_review_test.csv')

X_train = train_data['review'].values
y_train = train_data['sentiment'].values

X_test = test_data['review'].values
y_test = test_data['sentiment'].values

# Tokenization by whitespace
def tokenizeBySpace(text):
    return text.split()

# Full tokenization using nltk's word_tokenize
def tokenizeFull(text):
    return nltk.tokenize.word_tokenize(text)

# Lowercasing
def lowerCase(text):
    return text.lower()

# Remove stopwords
nltk.download('stopwords')
def removeStopwords(text):
    stop_words = set(nltk.corpus.stopwords.words('english'))
    return ' '.join([word for word in text.split() if word not in stop_words])

# Lemmatization
def lemitaizeText(text):
    lemmatizer = nltk.stem.WordNetLemmatizer()
    return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

def remove_punctuation(text):
    return re.sub(r'[^\w\s]', '', text)



[nltk_data] Downloading package stopwords to /Users/ovak/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [10]:
### TAKEN FROM MY PA1 PIECE OF CODE

def preprocess_data(reviews_df, N, frequency='most', strength='lightly'):
    
    # Preprocess reviews and return processed texts and vocabulary.
    
    processed_reviews = []
    all_tokens = []
    start_time = time.time()
    for review in reviews_df:
        text = str(review)
        
        # Apply lowercase, common for both strengths
        text = lowerCase(text)

        # need to remove punctuation here because otherwise 
        # tokens will have punctuations for most frequency selections
        text = remove_punctuation(text)
        
        # Tokenize based on strength
        if strength == 'lightly':
            tokens = tokenizeBySpace(text)
        else:  # strength == 'fully'
            tokens = tokenizeFull(text)
            
            text_no_stop = removeStopwords(' '.join(tokens))
            
            text_lemmatized = lemitaizeText(text_no_stop)
            
            tokens = text_lemmatized.split()
        
        # Store processed text and tokens
        processed_reviews.append(' '.join(tokens))
        all_tokens.extend(tokens)
    
    # Count token frequencies
    token_counts = Counter(all_tokens)
    
    # Select N tokens based on frequency
    if frequency == 'most':
        # Get the N most common tokens
        selected_tokens = [token for token, count in token_counts.most_common(N)]
    else: 
        # Get the N least common tokens
        selected_tokens = [token for token, count in token_counts.most_common()[:-N-1:-1]] 
    
    vocabulary = selected_tokens
    print(f" {strength}_preprocessed_{frequency}_{N} is ready! (Time: {start_time})")
    return processed_reviews, vocabulary


In [11]:
def get_bow_features(processed_reviews, vocabulary):
    
    # Convert processed reviews into Bag-of-Words features using a fixed vocabulary.
    vectorizer = CountVectorizer(vocabulary=vocabulary)
    X = vectorizer.fit_transform(processed_reviews)
    return X

In [12]:
fully_preprocessed_most_500, vac = preprocess_data(X_train, N=500, frequency='most', strength='fully')

# Train/test split
X_train_bow = get_bow_features(fully_preprocessed_most_500, vac)

# Preprocess test set using the SAME preprocessing function
fully_preprocessed_test, _ = preprocess_data(X_test, N = len(vac), frequency='most', strength='fully')
X_test_bow = get_bow_features(fully_preprocessed_test, vac)

print(X_train_bow.shape)
print(X_test_bow.shape)
print(y_train.shape)
print(y_test.shape)


 fully_preprocessed_most_500 is ready! (Time: 1764586600.229246)
 fully_preprocessed_most_500 is ready! (Time: 1764586601.481833)
(14675, 500)
(1675, 500)
(14675,)
(1675,)


In [13]:
tokenized_train = train_data['review'].apply(preprocess_review).tolist()
tokenized_test = test_data['review'].apply(preprocess_review).tolist()

w2v_model = gensim.models.Word2Vec(
    sentences=tokenized_train,
    vector_size=100,
    window=5,
    min_count=1,   # keep all words for now
    sg=1,
    epochs=10
)
all_tokens = [t for r in tokenized_train for t in r]
top_100_tokens = [t for t,_ in Counter(all_tokens).most_common(100)]

# Convert review to average vector of tokens in top 100
import numpy as np

def review_to_w2v_avg(tokens, model, top_tokens, vector_size=100):
    vecs = [model.wv[t] for t in tokens if t in top_tokens and t in model.wv]
    if len(vecs) == 0:
        return np.zeros(vector_size)
    return np.mean(vecs, axis=0)

X_train_w2v = np.array([review_to_w2v_avg(r, w2v_model, top_100_tokens) for r in tokenized_train])
X_test_w2v = np.array([review_to_w2v_avg(r, w2v_model, top_100_tokens) for r in tokenized_test])

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [14]:
# --- BoW classifier ---
clf_bow = LogisticRegression(max_iter=500)
clf_bow.fit(X_train_bow, y_train)
y_pred_bow = clf_bow.predict(X_test_bow)

# --- Word2Vec classifier ---
clf_w2v = LogisticRegression(max_iter=500)
clf_w2v.fit(X_train_w2v, y_train)
y_pred_w2v = clf_w2v.predict(X_test_w2v)

In [15]:
def evaluate(y_true, y_pred, name="Model"):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    print(f"{name} Performance:")
    print(f"Accuracy: {acc:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}\n")

evaluate(y_test, y_pred_bow, "Bag-of-Words Classifier")
evaluate(y_test, y_pred_w2v, "Word2Vec Classifier")

Bag-of-Words Classifier Performance:
Accuracy: 0.8519, Precision: 0.8799, Recall: 0.8598, F1: 0.8697

Word2Vec Classifier Performance:
Accuracy: 0.7982, Precision: 0.8438, Recall: 0.7965, F1: 0.8194



### Q1.D - discussion

**Performance Comparison:**

Both classifiers were trained using Logistic Regression on the same training and test sets. The key difference lies in the feature representation:

**Bag-of-Words (BOW) Classifier:**
- Uses frequency counts of the top 100 most frequent tokens
- Simple and interpretable representation
- Captures word presence/frequency but ignores semantic relationships
- Results: [Insert your results here after running]

**Word2Vec Classifier:**
- Uses averaged word embeddings from the top 100 most frequent tokens
- Captures semantic similarity between words
- Dense representation (100-dimensional vectors vs. sparse 100-dimensional counts)
- Results: [Insert your results here after running]

**Discussion Points:**

1. **Semantic Understanding**: Word2Vec should theoretically perform better because it captures semantic relationships (e.g., "excellent" and "great" have similar vectors), while BOW treats them as independent features.

2. **Information Loss**: Averaging Word2Vec vectors can lose important sentiment information, especially when positive and negative words appear together in a review.

3. **Feature Space**: BOW uses sparse binary/count features, while Word2Vec uses dense continuous features that may generalize better with limited data.

4. **Vocabulary Coverage**: Restricting to top 100 tokens might hurt Word2Vec more, as it benefits from a richer vocabulary to leverage semantic relationships.

5. **Training Data Size**: With the current dataset size, simpler BOW representation might perform comparably or even better than Word2Vec due to less complexity and overfitting.

**Conclusion:**
[Add your conclusion based on the actual results - which model performed better and why?]

## Q2 - Neural Networks for Binary Classification (50 points)

For this task, you will use the `user_review_train.csv` and `user_review_test.csv` files to train two neural network models for the binary classification of user reviews and compare their performances. You are expected to train RNN (part A - 20 points) and TextCNN (part B - 20 points) models, and report the following: 

- Confusion matrix of both models
- Time it took to train both models
- Accuracy, precision, recall, and F1-score of both models
- Other metrics you think are important

Finally (part C), you should discuss the performance of the models according to your reported results. Try to analyse the models in terms of pros and cons of using each one.

#### Notes and tips

- For the embedding layers of the models, you are free to use word embedding methods or leave them randomly initialised. Similarly, you can use word-based or character-based embeddings. However, make sure to explain your decisions.
- You are expected to use `tensorflow` for your implementations, but you can use other libraries if you already have a working setup.

In [70]:
# Q2.A - implementation of RNN
# you can add cells below if needed
train_data = pd.read_csv('data/user_review_train.csv')
test_data = pd.read_csv('data/user_review_test.csv')

X_train = train_data['review'].values
y_train = train_data['sentiment'].values

X_test = test_data['review'].values
y_test = test_data['sentiment'].values


In [71]:


# ensure we've read the data correctly (using head() to avoid too much output)
print(train_data.head())
print(test_data.head())

def clean_text(t):
    t = t.lower()
    t = re.sub(r'\s+', ' ', t).strip()
    return t

# Preprocess the text data. 
# It is different then previous state of art NLP preprocessing such as stop word removal and lemmatization. 
# Because RNNs can learn from the full sequences, we only lowercase and remove extra spaces here.
X_train = [clean_text(x) for x in X_train]
X_test = [clean_text(x) for x in X_test]

   sentiment                                             review
0          0  Worst mobile i have bought ever, Battery is dr...
1          0  The worst phone everThey have changed the last...
2          0  Only I'm telling don't buyI'm totally disappoi...
3          0                    The battery level has worn down
4          0  It's over hitting problems...and phone hanging...
   sentiment                                             review
0          0  I am using this product from last 37 days it i...
1          0  Not for jio... U need the volte app for callim...
2          0  For these... particular... features..I think p...
3          0  Phone is no doubt good, but battery should hav...
4          0                                             Gd phn


In [72]:
# Tokenize and pad sequences

# Fit tokenizer
tokenizer = Tokenizer(oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

# Dynamic vocab size
vocab_size = len(tokenizer.word_index) + 1
print("VOCAB SIZE =", vocab_size)

# Compute dynamic max_len (95th percentile) to eliminate outliers in training data
sequences = tokenizer.texts_to_sequences(X_train)
lengths = [len(s) for s in sequences]
max_len = int(np.percentile(lengths, 95))
print("MAX_LEN =", max_len)

# choose embedding dimension as 64 for this example. No special reason for this value other than being a power of 2 and not too large.
embedding_dim = 64

#those paramters will be used for RNN and TextCNN models
print("EMBEDDING DIM =", embedding_dim)

# Important Note: I prefer to calculate vocab_size and max_len dynamically from the training data
# rather than using fixed values although using hard coded values are used in simple example on internet. 
# This allows the model to adapt better to the actual data distribution I hope :)

# Pad using dynamic max_len
X_train_pad = pad_sequences(sequences, maxlen=max_len, padding="post")
X_test_pad = pad_sequences(
    tokenizer.texts_to_sequences(X_test),
    maxlen=max_len,
    padding="post"
)

VOCAB SIZE = 13008
MAX_LEN = 69
EMBEDDING DIM = 64


In [73]:
# Build RNN model now 


# define the RNN model. Keep it simple with one SimpleRNN layer. Hidden layer size is chosen as 64.
# since we will use this model sentiment analysis, final layer has 1 neuron with sigmoid activation. It is (binary classification).
cnnModel = Sequential([
    Embedding(vocab_size, embedding_dim, input_length=max_len),
    SimpleRNN(64),
    Dense(1, activation='sigmoid')
])

# Build the model by specifying input shape. 
# I need to put this here to show the model summary later. If not called here, model.summary() will give eöpty.
cnnModel.build(input_shape=(None, max_len))

# classic complile step for binary classification
cnnModel.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

cnnModel.summary()


/Users/ovak/Documents/dev/METU-CENG/CENG463/PA1/student_pack/.venv/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_11 (Embedding)        │ (None, 69, 64)         │       832,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_6 (SimpleRNN)        │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 840,833 (3.21 MB)

 Trainable params: 840,833 (3.21 MB)

 Non-trainable params: 0 (0.00 B)

In [74]:
# train the rnn model

history = cnnModel.fit(
    X_train_pad,
    y_train,
    validation_data=(X_test_pad, y_test),
    epochs=5,
    batch_size=32
)

Epoch 1/5
459/459 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - accuracy: 0.5312 - loss: 0.6865 - val_accuracy: 0.5982 - val_loss: 0.6330
Epoch 2/5
459/459 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6699 - loss: 0.6175 - val_accuracy: 0.6645 - val_loss: 0.5672
Epoch 3/5
459/459 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.5640 - loss: 0.6581 - val_accuracy: 0.6275 - val_loss: 0.6588
Epoch 4/5
459/459 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6370 - loss: 0.6292 - val_accuracy: 0.7696 - val_loss: 0.5115
Epoch 5/5
459/459 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.7704 - loss: 0.5139 - val_accuracy: 0.8072 - val_loss: 0.4693


In [75]:
# test

loss, acc = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", acc)

y_pred_proba = model.predict(X_test_pad)
y_pred = (y_pred_proba > 0.5).astype(int)

# compute F1
f1 = f1_score(y_test, y_pred)
print("F1 Score:", f1)


53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4304 - loss: 0.6944
Test Accuracy: 0.4304477572441101
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
F1 Score: 0.1660839160839161


In [76]:
# Q2.B - implementation of TextCNN
# you can add cells below if needed


input_layer = Input(shape=(max_len,))

embedding = Embedding(vocab_size, embedding_dim, input_length=max_len)(input_layer)

# parallel convolution layers with different kernel sizes
conv_3 = Conv1D(filters=128, kernel_size=3, activation='relu')(embedding)
conv_4 = Conv1D(filters=128, kernel_size=4, activation='relu')(embedding)
conv_5 = Conv1D(filters=128, kernel_size=5, activation='relu')(embedding)

# global max pooling
pool_3 = GlobalMaxPooling1D()(conv_3)
pool_4 = GlobalMaxPooling1D()(conv_4)
pool_5 = GlobalMaxPooling1D()(conv_5)

# concatenate
concat = Concatenate()([pool_3, pool_4, pool_5])

dropout = Dropout(0.5)(concat)

output = Dense(1, activation='sigmoid')(dropout)

model = Model(inputs=input_layer, outputs=output)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


/Users/ovak/Documents/dev/METU-CENG/CENG463/PA1/student_pack/.venv/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, 69)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_12        │ (None, 69, 64)    │    832,512 │ input_layer_9[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_15 (Conv1D)  │ (None, 67, 128)   │     24,704 │ embedding_12[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_16 (Conv1D)  │ (None, 66, 128)   │     32,896 │ embedding_12[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_17 (Conv1D)  │ (None, 65, 128)   │     41,088 │ embedding_12[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_15[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_16[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_17[0][0]   │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_5       │ (None, 384)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 384)       │          0 │ concatenate_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │        385 │ dropout_5[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 931,585 (3.55 MB)

 Trainable params: 931,585 (3.55 MB)

 Non-trainable params: 0 (0.00 B)

In [77]:
loss, acc = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", acc)

# predict
y_pred_proba = model.predict(X_test_pad)
y_pred = (y_pred_proba > 0.5).astype(int)

# f1
f1 = f1_score(y_test, y_pred)
print("F1 Score:", f1)

53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5116 - loss: 0.6931
Test Accuracy: 0.511641800403595
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
F1 Score: 0.4749679075738126


### Q2.C - discussion

Write your discussion here.